Wednesday, hands on: the protect list, the flag and the plan line, solved

> "Ties matter. If two members spent the same, I want them ranked the same, and I want to know
> how many made the top fifty, not forty-nine because of a tie."

Released at close of session. Step 3 has a business answer rather than a technical one, and
no check can tell you that.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
kit.flow(["rank", "meet the tie", "choose a rule", "flag the fallers", "accumulate"],
         lit=[0], title="Where you are")

## 1. Q2 revenue per Retail-Plus member

One row per member, carrying their Q2 revenue. This is still `GROUP BY` territory.

In [2]:
q2 = kit.sql("""SELECT o.customer_id, sum(o.amount) AS revenue
       FROM orders o JOIN customers c USING (customer_id)
       WHERE o.quarter='Q2' AND c.segment='Retail-Plus'
       GROUP BY o.customer_id""", conn=conn)
kit.check("more than fifty members to rank", len(q2) > 50, f"{len(q2)} members")

## 2. Three rankings side by side

Add `row_number`, `rank` and `dense_rank` over the same ordering. Look at positions 46 to 54
rather than the top ten; the top of a list is never where the argument happens.

In [3]:
edge = kit.sql("""WITH q2 AS (SELECT o.customer_id, sum(o.amount) AS revenue
       FROM orders o JOIN customers c USING (customer_id)
       WHERE o.quarter='Q2' AND c.segment='Retail-Plus'
       GROUP BY o.customer_id),
            r AS (SELECT customer_id, revenue,
                         row_number() OVER (ORDER BY revenue DESC) AS rn,
                         rank()       OVER (ORDER BY revenue DESC) AS rk,
                         dense_rank() OVER (ORDER BY revenue DESC) AS dr
                  FROM q2)
       SELECT rn, rk, dr, revenue FROM r WHERE rn BETWEEN 46 AND 54 ORDER BY rn""", conn=conn)
kit.table(list(edge[0]) if edge else ["rn"], [list(r.values()) for r in edge],
          caption="The boundary, where the tie lives")
kit.check("nine rows around the boundary", len(edge) == 9, f"{len(edge)} rows")

rn,rk,dr,revenue
46,46,46,3600.00
47,47,47,3540.00
48,48,48,3480.00
49,48,48,3480.00
50,50,49,3350.00
51,50,49,3350.00
52,52,50,3200.00
53,53,51,3150.00
54,54,52,3110.00


## 3. How many names does each rule ship?

Count how many rows survive a filter at fifty under each of the three functions.

In [4]:
counts = kit.sql("""WITH q2 AS (SELECT o.customer_id, sum(o.amount) AS revenue
       FROM orders o JOIN customers c USING (customer_id)
       WHERE o.quarter='Q2' AND c.segment='Retail-Plus'
       GROUP BY o.customer_id),
            r AS (SELECT row_number() OVER (ORDER BY revenue DESC) rn,
                         rank()       OVER (ORDER BY revenue DESC) rk,
                         dense_rank() OVER (ORDER BY revenue DESC) dr FROM q2)
       SELECT count(*) FILTER (WHERE rn<=50) AS by_row_number,
              count(*) FILTER (WHERE rk<=50) AS by_rank,
              count(*) FILTER (WHERE dr<=50) AS by_dense_rank FROM r""", conn=conn)[0]
print(dict(counts))
kit.check("the three rules disagree", len(set(counts.values())) == 3, str(dict(counts)))
kit.decision_ladder(["ROW_NUMBER, which drops one of a tied pair",
                     "DENSE_RANK, which moves the cut further down",
                     "RANK, which keeps both tied members"],
                    cut_at=2, title="Which one he asked for")

{'by_row_number': 50, 'by_rank': 51, 'by_dense_rank': 52}


## 4. The protect list

Top fifty per segment under the rule the head of Retail-Plus asked for. Remember that a window
cannot be filtered in `WHERE`, so compute it inside and filter outside.

In [5]:
protect = kit.sql("""WITH q2 AS (SELECT c.segment, o.customer_id, sum(o.amount) AS revenue
                   FROM orders o JOIN customers c USING (customer_id)
                   WHERE o.quarter='Q2' GROUP BY c.segment, o.customer_id),
            r AS (SELECT segment, customer_id, revenue,
                         rank() OVER (PARTITION BY segment ORDER BY revenue DESC) AS pos
                  FROM q2)
       SELECT segment, count(*) AS names FROM r WHERE pos <= 50
       GROUP BY segment ORDER BY segment""", conn=conn)
kit.check("Retail-Plus ships fifty-one names",
          next((r for r in protect if r["segment"] == "Retail-Plus"))["names"] == 51, str(protect))

## 5. Falling two months running

Monthly spend per member, then two LAGs and a comparison. Decide before you write it what a
member with only one month of data should do to your flag.

In [6]:
falling = kit.sql("""WITH monthly AS (
           SELECT o.customer_id, date_trunc('month', o.order_date) AS mth, sum(o.amount) AS spend
           FROM orders o JOIN customers c USING (customer_id)
           WHERE o.quarter='Q2' AND c.segment='Retail-Plus'
           GROUP BY o.customer_id, date_trunc('month', o.order_date)),
            l AS (SELECT customer_id, mth, spend,
                         lag(spend,1) OVER (PARTITION BY customer_id ORDER BY mth) AS m1,
                         lag(spend,2) OVER (PARTITION BY customer_id ORDER BY mth) AS m2
                  FROM monthly)
       SELECT customer_id, m2 AS july, m1 AS august, spend AS september
       FROM l WHERE m2 > m1 AND m1 > spend ORDER BY customer_id""", conn=conn)
kit.check("three members fall in both steps", len(falling) == 3, f"{len(falling)}")
kit.vflow(["July", "August", "September"], lit=[2], title="Two steps down, not one")

## 6. The running total against plan

Weekly Q2 revenue, accumulating, beside the plan line accumulating. Give the window an order
that cannot tie, or the cumulative column can differ between runs.

In [7]:
plan = kit.sql("""WITH weekly AS (SELECT date_trunc('week', order_date) AS week_start,
                              sum(amount) AS revenue
                       FROM orders WHERE quarter='Q2' GROUP BY date_trunc('week', order_date))
       SELECT w.week_start::date AS week, w.revenue,
              sum(w.revenue) OVER (ORDER BY w.week_start) AS cumulative,
              sum(p.plan_revenue) OVER (ORDER BY w.week_start) AS plan_cumulative
       FROM weekly w LEFT JOIN plan_line p ON p.week_start = w.week_start::date
       ORDER BY w.week_start""", conn=conn)
kit.check("thirteen or fourteen weeks in the quarter", 13 <= len(plan) <= 14, f"{len(plan)}")
kit.ladder(["weekly revenue", "cumulative", "plan cumulative", "the gap between them"],
           lit=[3], title="What Meera reads")

## 7. The note

Four sentences: which tie rule you chose, the sentence from the head of Retail-Plus that
decided it, how many names the Retail-Plus list contains, and what you would say to a flagged
member who was on holiday in August.

The fourth sentence is the one that matters. A flag is a shortlist for a conversation rather
than a verdict.

In [8]:
kit.matrix(["the list", "the flag", "the plan line"],
           ["what it is", "what it is not"],
           [["fifty-one names", "a ranking of worth"],
            ["three members to call", "proof of anything"],
            ["ahead, then level", "a forecast"]],
           title="What you are handing over")
kit.check_summary()